# M.E.R.L.I.N. — Knowledge Distillation: MerlinNarrator-0.3B

## Goal
Distill Qwen 2.5-1.5B (teacher) into a custom ~300M param model (student).

| | Teacher | Student |
|---|---|---|
| Params | 1.54B | ~300M |
| Layers | 28 | 16 |
| Hidden | 1536 | 1024 |
| GGUF size | ~1.0 GB | ~200 MB |
| CPU tok/s | ~18 | ~45-60 |

## Two-Session Strategy (recommended for free Colab)

### Session A — Training only (~45-60 min GPU) ✅ START HERE
Set `SKIP_PHASE1 = True` in Cell 1b.
Uses the existing 724 gold samples directly.
Covers: Cell 1a → Cell 1b → Cell 2 → Cell 3 → Cell 4 (skip) → Cell 5 → Cell 6 → Cell 7 → Cell 8 → Cell 9 → Cell 10 → Cell 11 → Cell 12

### Session B — Data augmentation (optional, ~3-4h GPU)
Set `SKIP_PHASE1 = False` in Cell 1b.
Teacher generates ~3000-5000 English samples, saves to Drive.
Only needed if Session A benchmark metrics are insufficient.

## Quick Start
1. Runtime > Change runtime type > **T4 GPU**
2. Cell 1b: set `SKIP_PHASE1 = True` for Session A
3. Run All

In [ ]:
# Cell 1a — Install dependencies
# On Colab: torch is pre-installed. On local: install everything.
import subprocess, sys

def pip_install(*packages):
    """Cross-platform pip install (works on Colab + local Windows/Linux)."""
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q"] + list(packages))

try:
    import torch
    print(f"torch already installed: {torch.__version__}")
except ImportError:
    print("Installing torch (this may take a few minutes)...")
    pip_install("torch")

# Install remaining deps
pip_install("transformers", "datasets", "accelerate", "sentencepiece", "bitsandbytes", "langdetect")
print("All dependencies installed.")

In [ ]:
# Cell 1b — GPU check + Drive mount
# ============================================================
# CONFIG — Modify this before running
# ============================================================
SKIP_PHASE1 = True   # True = Session A (~45-60 min), False = Session B (~3-4h)
# ============================================================

import os
# Must be set BEFORE first CUDA allocation — reduces allocator fragmentation
# (makes T4's 14.5 GB usable without "can't allocate X more GiB" failures)
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"

import torch, json, time, gc, re, shutil
from pathlib import Path

# Detect environment
try:
    from google.colab import drive
    IS_COLAB = True
except ImportError:
    IS_COLAB = False

# GPU verification
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_mem = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"GPU: {gpu_name} ({gpu_mem:.1f} GB)")
else:
    gpu_mem = 0
    print("WARNING: No GPU — go to Runtime > Change runtime type > T4 GPU")

print(f"PyTorch: {torch.__version__}")
print(f"Mode: {'Session A — training only (SKIP_PHASE1=True)' if SKIP_PHASE1 else 'Session B — full pipeline with data augmentation'}")
print(f"PYTORCH_ALLOC_CONF: {os.environ.get('PYTORCH_ALLOC_CONF', 'not set')}")

# Mount Google Drive or use local dir
if IS_COLAB:
    drive.mount('/content/drive')
    DRIVE_BASE = "/content/drive/MyDrive/M.E.R.L.I.N./distillation"
else:
    DRIVE_BASE = os.path.join(os.getcwd(), "distillation-output")
    print(f"Local mode: {DRIVE_BASE}")

os.makedirs(DRIVE_BASE, exist_ok=True)
for subdir in ["checkpoints", "dataset", "merged", "benchmark"]:
    os.makedirs(f"{DRIVE_BASE}/{subdir}", exist_ok=True)
print(f"Output base: {DRIVE_BASE}")

In [ ]:
# Cell 2 — Initialize Student model (custom Qwen2 config, ~300M params)
from transformers import Qwen2Config, Qwen2ForCausalLM, AutoTokenizer

# === PREREQUISITE CHECK ===
if not torch.cuda.is_available():
    raise RuntimeError(
        "\n" + "="*60 +
        "\nERROR: No GPU available!\n"
        "On Colab: Runtime > Change runtime type > T4 GPU\n"
        "Then: Runtime > Restart session\n"
        "Then re-run from Cell 1a.\n" +
        "="*60
    )
print(f"GPU ready: {torch.cuda.get_device_name(0)}")

print("Loading Qwen2.5-1.5B tokenizer (shared with student)...")
tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-1.5B-Instruct", trust_remote_code=True)

# CRITICAL: Qwen tokenizer has no pad_token by default
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.pad_token_id = tokenizer.eos_token_id
    print(f"  pad_token set to eos_token (id={tokenizer.pad_token_id})")

# Custom student config — Qwen2 architecture, reduced dimensions
student_config = Qwen2Config(
    hidden_size=1024,
    num_hidden_layers=16,
    num_attention_heads=8,
    num_key_value_heads=2,
    intermediate_size=5632,
    vocab_size=151936,
    max_position_embeddings=4096,
    rms_norm_eps=1e-6,
    rope_theta=1000000.0,
    tie_word_embeddings=True,
    hidden_act="silu",
    use_sliding_window=False,
    torch_dtype="bfloat16",
)

print("Initializing MerlinNarrator-0.3B from scratch...")
student = Qwen2ForCausalLM(student_config)

total_params = sum(p.numel() for p in student.parameters())
embed_params = sum(p.numel() for n, p in student.named_parameters() if "embed" in n)
print(f"Student: {total_params / 1e6:.1f}M total params ({(total_params - embed_params) / 1e6:.1f}M non-embedding)")
print(f"Config: {student_config.num_hidden_layers}L / {student_config.hidden_size}H / {student_config.num_attention_heads}A")

# GGUF compatibility dry-run
test_dir = "/content/student_test"
student.save_pretrained(test_dir)
tokenizer.save_pretrained(test_dir)
loaded_config = Qwen2Config.from_pretrained(test_dir)
assert loaded_config.model_type == "qwen2", "Config type mismatch!"
print("GGUF compatibility: OK (model_type=qwen2)")
shutil.rmtree(test_dir)

# Move to GPU
dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
student = student.to(dtype=dtype, device="cuda")
print(f"Student on GPU: {torch.cuda.memory_allocated() / 1e9:.2f} GB used")

In [ ]:
# Cell 3 — Load Teacher model (4-bit quantized for VRAM efficiency)
from transformers import AutoModelForCausalLM, BitsAndBytesConfig

print("Loading teacher Qwen 2.5-1.5B-Instruct (4-bit NF4)...")
t0 = time.time()

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16,
)

teacher = AutoModelForCausalLM.from_pretrained(
    "Qwen/Qwen2.5-1.5B-Instruct",
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)
teacher.eval()
for p in teacher.parameters():
    p.requires_grad = False

print(f"Teacher loaded in {time.time() - t0:.0f}s")
print(f"GPU after teacher: {torch.cuda.memory_allocated() / 1e9:.2f} GB")
print(f"GPU total used: {torch.cuda.memory_reserved() / 1e9:.2f} GB / {gpu_mem:.1f} GB")

In [ ]:
# Cell 4 — Load dataset
# If SKIP_PHASE1=True: load base dataset directly (724 samples)
# If SKIP_PHASE1=False: will augment in Cell 4b

augmented_path = f"{DRIVE_BASE}/dataset/merlin_distill_en.jsonl"

# Check resume first (augmented dataset from a previous Session B)
if os.path.exists(augmented_path):
    with open(augmented_path) as f:
        distill_samples = [json.loads(l) for l in f if l.strip()]
    print(f"RESUME: Found {len(distill_samples)} augmented samples on Drive — using them")
    SKIP_AUGMENTATION = True

elif SKIP_PHASE1:
    # Session A: load base dataset directly
    SKIP_AUGMENTATION = True
    search_paths = [
        f"{DRIVE_BASE}/dataset/merlin_full_v8.jsonl",
    ]
    if IS_COLAB:
        search_paths += [
            "/content/drive/MyDrive/M.E.R.L.I.N./merlin_full_v8.jsonl",
            "/content/drive/MyDrive/M.E.R.L.I.N/merlin_full_v8.jsonl",
        ]
    else:
        search_paths += [
            os.path.join(os.getcwd(), "data", "ai", "training", "merlin_full_v8.jsonl"),
            "c:/Users/PGNK2128/Godot-MCP/data/ai/training/merlin_full_v8.jsonl",
        ]

    found = False
    for src in search_paths:
        if os.path.exists(src):
            print(f"Dataset from: {src}")
            with open(src) as f:
                distill_samples = [json.loads(l) for l in f if l.strip()]
            found = True
            break

    if not found:
        if IS_COLAB:
            from google.colab import files
            print("Upload merlin_full_v8.jsonl (from data/ai/training/):")
            uploaded = files.upload()
            for name in uploaded:
                if name.endswith(".jsonl"):
                    import io
                    distill_samples = [json.loads(l) for l in uploaded[name].decode().splitlines() if l.strip()]
                    break
        else:
            raise FileNotFoundError(f"Place merlin_full_v8.jsonl in one of:\n" + "\n".join(search_paths))

    print(f"Session A: {len(distill_samples)} base samples loaded (no augmentation)")

else:
    # Session B: will augment in Cell 4b
    SKIP_AUGMENTATION = False
    dataset_path = "/content/merlin_full_v8.jsonl" if IS_COLAB else "merlin_full_v8.jsonl"
    search_paths = [
        f"{DRIVE_BASE}/dataset/merlin_full_v8.jsonl",
        "/content/drive/MyDrive/M.E.R.L.I.N./merlin_full_v8.jsonl",
        "c:/Users/PGNK2128/Godot-MCP/data/ai/training/merlin_full_v8.jsonl",
    ]
    for src in search_paths:
        if os.path.exists(src):
            shutil.copy2(src, dataset_path)
            print(f"Dataset from: {src}")
            break
    else:
        if IS_COLAB:
            from google.colab import files
            print("Upload merlin_full_v8.jsonl:")
            uploaded = files.upload()
            for name in uploaded:
                if name.endswith(".jsonl"):
                    with open(dataset_path, "wb") as f:
                        f.write(uploaded[name])
                    break

    with open(dataset_path) as f:
        base_samples = [json.loads(l) for l in f if l.strip()]
    distill_samples = []  # Will be populated in Cell 4b
    print(f"Session B: {len(base_samples)} base samples — augmentation will run in Cell 4b")

print(f"distill_samples ready: {len(distill_samples)}")

In [ ]:
# Cell 4b — Phase 1: Generate English augmented dataset (Session B only)
# SKIPPED automatically when SKIP_PHASE1=True or augmented dataset already on Drive

if SKIP_AUGMENTATION:
    print(f"Cell 4b SKIPPED — using {len(distill_samples)} samples already loaded")
    print("(Set SKIP_PHASE1=False for a full augmentation run in Session B)")
else:
    import random
    print("=" * 60)
    print(f"PHASE 1: Augmentation — Session B ({len(base_samples)} → ~3000+ EN samples)")
    print("=" * 60)

    EN_PRIMER = (
        "You are M.E.R.L.I.N. \u2014 Memory Eternal of Recounted Legends and Incarnate Narratives. "
        "Born from the beliefs of men, assembled through centuries of tales. "
        "A wild druid of Broceliande, guide of the Traveller through Celtic biomes. "
        "Vocabulary: mist, stone, ogham, nemeton, sidhe, dolmen, korrigans, rune, breath. "
        "English only. Short, evocative sentences."
    )
    FORMAT_INST = (
        "\nOBLIGATORY FORMAT: Write a narrative paragraph, then exactly 3 choices:\n"
        "A) VERB \u2014 description\nB) VERB \u2014 description\nC) VERB \u2014 description"
    )
    BIOMES = ["broceliande_forest", "korrigan_marshes", "sidhe_mountains", "carnac_coast",
              "crystal_cave", "menhir_plains", "avalon_lake", "druid_village",
              "nemeton_ruins", "ogham_sanctuary"]
    THEMES = ["sacred spring", "Samhain night", "mystical creature", "physical trial",
              "moral dilemma", "ancestral meeting", "magical storm", "ancient ritual",
              "betrayal", "revelation", "harvest moon", "winter solstice",
              "faerie bargain", "lost traveller", "cursed relic"]
    ACTS = ["Act I (beginning)", "Act II (confrontation)", "Act III (resolution)"]
    TRIADE = [
        "Body=low Soul=balanced World=balanced", "Body=balanced Soul=low World=balanced",
        "Body=balanced Soul=balanced World=low", "Body=high Soul=balanced World=low",
        "Body=low Soul=high World=balanced", "Body=balanced Soul=low World=high",
    ]
    CHOICE_PATTERN = re.compile(r'^[A-C][).]\s+[A-Z]', re.MULTILINE)

    def generate_sample(system_prompt, user_prompt, temperature=0.7, max_tokens=250):
        chatml = (
            f"<|im_start|>system\n{system_prompt}<|im_end|>\n"
            f"<|im_start|>user\n{user_prompt}<|im_end|>\n"
            f"<|im_start|>assistant\n"
        )
        inputs = tokenizer(chatml, return_tensors="pt").to(teacher.device)
        with torch.no_grad():
            out = teacher.generate(
                **inputs, max_new_tokens=max_tokens, temperature=temperature,
                top_p=0.9, repetition_penalty=1.3, do_sample=True,
            )
        text = tokenizer.decode(out[0], skip_special_tokens=False)
        answer = text.split("<|im_start|>assistant\n")[-1].split("<|im_end|>")[0].strip()
        del inputs, out
        return answer

    def score_sample(text):
        score = 0.0
        if len(CHOICE_PATTERN.findall(text)) >= 3:
            score += 0.4
        if 50 <= len(text.split()) <= 500:
            score += 0.3
        en_markers = {"the", "and", "you", "your", "through", "with", "into"}
        if len(set(text.lower().split()) & en_markers) >= 2:
            score += 0.3
        return score

    distill_samples = []
    t0 = time.time()

    # Phase 1a: translate existing gold samples
    print(f"\n--- Phase 1a: Translating {len(base_samples)} gold samples to EN ---")
    for idx, sample in enumerate(base_samples):
        msgs = sample.get("messages", [])
        user_content = msgs[1].get("content", "") if len(msgs) > 1 else ""
        system = EN_PRIMER + FORMAT_INST
        user = f"Generate a narrative card in English. Context: {user_content[:200]}"
        answer = generate_sample(system, user, temperature=0.65)
        if score_sample(answer) >= 0.5:
            distill_samples.append({"messages": [
                {"role": "system", "content": EN_PRIMER},
                {"role": "user", "content": user},
                {"role": "assistant", "content": answer},
            ]})
        if (idx + 1) % 50 == 0:
            elapsed = time.time() - t0
            eta = elapsed / (idx + 1) * (len(base_samples) - idx - 1)
            print(f"  [{idx+1}/{len(base_samples)}] {len(distill_samples)} kept | {elapsed/60:.0f}m elapsed, ~{eta/60:.0f}m ETA")
            gc.collect(); torch.cuda.empty_cache()
            # Save intermediate to Drive
            with open(augmented_path, "w") as f:
                for s in distill_samples:
                    f.write(json.dumps(s, ensure_ascii=False) + "\n")

    # Phase 1b: combinatorial generation
    print(f"\n--- Phase 1b: Generating new combinatorial samples ---")
    target_new = 2000
    combos = [(b, t, a, tr) for b in BIOMES for t in THEMES[:8] for a in ACTS for tr in TRIADE]
    random.seed(42)
    random.shuffle(combos)
    combos = combos[:target_new]
    temps = [0.5, 0.65, 0.8]

    for cidx, (biome, theme, act, triade) in enumerate(combos):
        temp = temps[cidx % len(temps)]
        system = EN_PRIMER + FORMAT_INST + f"\n{act}."
        user = f"Card {cidx+1}. Biome: {biome}. Theme: {theme}. {triade}."
        answer = generate_sample(system, user, temperature=temp)
        if score_sample(answer) >= 0.5:
            distill_samples.append({"messages": [
                {"role": "system", "content": EN_PRIMER},
                {"role": "user", "content": user},
                {"role": "assistant", "content": answer},
            ]})
        if (cidx + 1) % 200 == 0:
            elapsed = time.time() - t0
            eta = elapsed / (cidx + 1) * (len(combos) - cidx - 1)
            print(f"  [{cidx+1}/{len(combos)}] {len(distill_samples)} total | ~{eta/60:.0f}m ETA")
            with open(augmented_path, "w") as f:
                for s in distill_samples:
                    f.write(json.dumps(s, ensure_ascii=False) + "\n")
            gc.collect(); torch.cuda.empty_cache()

    # Final save
    random.shuffle(distill_samples)
    with open(augmented_path, "w") as f:
        for s in distill_samples:
            f.write(json.dumps(s, ensure_ascii=False) + "\n")
    print(f"\nPHASE 1 DONE: {len(distill_samples)} samples in {(time.time()-t0)/3600:.1f}h")
    print(f"Saved to Drive: {augmented_path}")
    print("Re-run Session A with SKIP_PHASE1=True to train on this augmented dataset.")

In [ ]:
# Cell 5 — Distillation configuration
# OOM fix applied: batch_size 8→2, grad_accum 2→8, max_seq_len 512→384
# Logit tensor at batch=2 seq=384: 2×384×151936×2B ≈ 234 MB  (was 1.2 GB at batch=8 seq=512)

DISTILL_CONFIG = {
    # Distillation loss
    "temperature": 3.0,       # Softens teacher distribution
    "alpha": 0.3,             # 0.3 hard + 0.7 soft
    "top_k_logits": 10,       # Top-K for KL divergence
    # Training — reduced for T4 VRAM
    "epochs": 5,
    "batch_size": 2,          # 8→2: logit tensors 1.2 GB → 234 MB per pass
    "grad_accum": 8,          # 2→8: effective batch = 16 (unchanged)
    "lr": 5e-5,
    "warmup_ratio": 0.1,
    "weight_decay": 0.01,
    "max_seq_len": 384,       # 512→384: ~25% less logit + activation memory
    # Checkpoints
    "save_steps": 100,
    "eval_split": 0.1,
    # Output
    "output_dir": "/content/distill-output",
    "drive_checkpoint_dir": f"{DRIVE_BASE}/checkpoints",
}

effective_batch = DISTILL_CONFIG["batch_size"] * DISTILL_CONFIG["grad_accum"]
steps_per_epoch = int(len(distill_samples) * (1 - DISTILL_CONFIG["eval_split"]) / effective_batch)
total_steps = steps_per_epoch * DISTILL_CONFIG["epochs"]

print("Distillation Configuration:")
print(f"  Loss:    {DISTILL_CONFIG['alpha']:.0%} hard + {1-DISTILL_CONFIG['alpha']:.0%} soft  (T={DISTILL_CONFIG['temperature']}, top-{DISTILL_CONFIG['top_k_logits']})")
print(f"  Dataset: {len(distill_samples)} samples  ({DISTILL_CONFIG['eval_split']:.0%} eval)")
print(f"  Batch:   {DISTILL_CONFIG['batch_size']} × {DISTILL_CONFIG['grad_accum']} accum = {effective_batch} effective")
print(f"  Seq len: {DISTILL_CONFIG['max_seq_len']} tokens  (was 512)")
print(f"  Steps:   {steps_per_epoch}/epoch × {DISTILL_CONFIG['epochs']} epochs = {total_steps} optimizer steps")
print(f"  Estimated: ~{total_steps * 15 // 60} min on T4  (~15s/optimizer-step)")

In [ ]:
# Cell 6 — DistillationTrainer class
import torch.nn.functional as F
from torch.utils.data import Dataset as TorchDataset, DataLoader
from IPython.display import clear_output


class ChatMLDataset(TorchDataset):
    """Tokenize ChatML messages for distillation."""

    def __init__(self, samples, tokenizer, max_len=512):
        self.encodings = []
        self.pad_token_id = tokenizer.pad_token_id
        for s in samples:
            msgs = s.get("messages", [])
            text = ""
            for msg in msgs:
                text += f"<|im_start|>{msg['role']}\n{msg['content']}<|im_end|>\n"
            enc = tokenizer(
                text, max_length=max_len, truncation=True,
                padding="max_length", return_tensors="pt",
            )
            self.encodings.append({
                "input_ids": enc["input_ids"].squeeze(0),
                "attention_mask": enc["attention_mask"].squeeze(0),
            })

    def __len__(self):
        return len(self.encodings)

    def __getitem__(self, idx):
        return self.encodings[idx]


def distillation_loss(student_logits, teacher_logits, labels, attention_mask,
                      temperature=3.0, alpha=0.3, top_k=10, pad_token_id=None):
    """Combined hard + soft label distillation loss."""
    # Shift for causal LM: predict next token
    shift_logits_s = student_logits[:, :-1, :].contiguous()
    shift_logits_t = teacher_logits[:, :-1, :].contiguous()
    shift_labels = labels[:, 1:].contiguous()
    shift_mask = attention_mask[:, 1:].contiguous().float()

    # Hard label loss (standard cross-entropy, masked)
    ignore_idx = pad_token_id if pad_token_id is not None else -100
    hard_loss = F.cross_entropy(
        shift_logits_s.view(-1, shift_logits_s.size(-1)),
        shift_labels.view(-1),
        ignore_index=ignore_idx,
        reduction="mean",
    )

    # Soft label loss (top-K KL divergence with temperature)
    with torch.no_grad():
        teacher_top_vals, teacher_top_idx = shift_logits_t.topk(top_k, dim=-1)
        teacher_probs = F.softmax(teacher_top_vals / temperature, dim=-1)

    # Gather student logits at teacher's top-K positions
    student_top_logits = torch.gather(shift_logits_s, -1, teacher_top_idx)
    student_log_probs = F.log_softmax(student_top_logits / temperature, dim=-1)

    # KL divergence (token-level, masked)
    kl_per_token = F.kl_div(student_log_probs, teacher_probs, reduction="none").sum(dim=-1)
    kl_masked = (kl_per_token * shift_mask).sum() / shift_mask.sum().clamp(min=1)
    soft_loss = kl_masked * (temperature ** 2)

    return alpha * hard_loss + (1 - alpha) * soft_loss, hard_loss.item(), soft_loss.item()


print("DistillationTrainer ready.")
print(f"Loss function: {DISTILL_CONFIG['alpha']:.0%} CE + {1-DISTILL_CONFIG['alpha']:.0%} KL(T={DISTILL_CONFIG['temperature']}, top-{DISTILL_CONFIG['top_k_logits']})")

In [ ]:
# Cell 7 — TRAIN (distillation main loop)
# OOM fixes: gradient_checkpointing + early deletion of teacher logits before backward
from torch.optim import AdamW
import random, math

# === Prepare datasets ===
print("Tokenizing dataset...")
random.shuffle(distill_samples)
split_idx = int(len(distill_samples) * (1 - DISTILL_CONFIG["eval_split"]))
train_samples = distill_samples[:split_idx]
eval_samples = distill_samples[split_idx:]

train_dataset = ChatMLDataset(train_samples, tokenizer, DISTILL_CONFIG["max_seq_len"])
eval_dataset = ChatMLDataset(eval_samples, tokenizer, DISTILL_CONFIG["max_seq_len"])
print(f"Train: {len(train_dataset)} | Eval: {len(eval_dataset)}")

train_loader = DataLoader(train_dataset, batch_size=DISTILL_CONFIG["batch_size"], shuffle=True, drop_last=True)
eval_loader = DataLoader(eval_dataset, batch_size=DISTILL_CONFIG["batch_size"], shuffle=False)

# === Optimizer + Scheduler ===
optimizer = AdamW(student.parameters(), lr=DISTILL_CONFIG["lr"], weight_decay=DISTILL_CONFIG["weight_decay"])
total_opt_steps = len(train_loader) // DISTILL_CONFIG["grad_accum"] * DISTILL_CONFIG["epochs"]
warmup_steps = int(total_opt_steps * DISTILL_CONFIG["warmup_ratio"])

def lr_lambda(step):
    if step < warmup_steps:
        return step / max(warmup_steps, 1)
    progress = (step - warmup_steps) / max(total_opt_steps - warmup_steps, 1)
    return max(0.1, 0.5 * (1.0 + math.cos(math.pi * progress)))

scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)

# === Resume support ===
resume_path = f"{DISTILL_CONFIG['drive_checkpoint_dir']}/latest_checkpoint.pt"
start_epoch = 0
global_step = 0
best_eval_loss = float("inf")
PAD_ID = tokenizer.pad_token_id

if os.path.exists(resume_path):
    print(f"Resuming from {resume_path}...")
    ckpt = torch.load(resume_path, map_location="cuda", weights_only=False)
    student.load_state_dict(ckpt["student_state_dict"])
    optimizer.load_state_dict(ckpt["optimizer_state_dict"])
    scheduler.load_state_dict(ckpt["scheduler_state_dict"])
    start_epoch = ckpt["epoch"]
    global_step = ckpt["global_step"]
    best_eval_loss = ckpt.get("best_eval_loss", float("inf"))
    print(f"Resumed: epoch {start_epoch}, step {global_step}, best_eval_loss {best_eval_loss:.4f}")
    del ckpt; gc.collect(); torch.cuda.empty_cache()

# === OOM FIX 1: Gradient checkpointing ===
# Recompute activations during backward instead of storing them.
# Saves ~40% activation memory at ~20% compute overhead (worth it on T4).
student.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
print("Gradient checkpointing: ENABLED (saves ~40% activation VRAM)")

# === Training loop ===
os.makedirs(DISTILL_CONFIG["output_dir"], exist_ok=True)
student.train()
teacher.eval()

gpu_total = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"\n{'='*60}")
print(f"DISTILLATION: {total_opt_steps} optimizer steps, {DISTILL_CONFIG['epochs']} epochs")
print(f"GPU budget: {torch.cuda.memory_allocated()/1e9:.1f} GB used / {gpu_total:.1f} GB total")
print(f"{'='*60}")

train_start = time.time()
running_loss = 0.0
running_hard = 0.0
running_soft = 0.0
log_interval = 10

for epoch in range(start_epoch, DISTILL_CONFIG["epochs"]):
    epoch_loss = 0.0
    optimizer.zero_grad()

    for batch_idx, batch in enumerate(train_loader):
        input_ids = batch["input_ids"].to("cuda")
        attention_mask = batch["attention_mask"].to("cuda")

        # === OOM FIX 2: Extract logits then immediately delete model outputs ===

        # Student forward (activations recomputed during backward via checkpointing)
        student_out = student(input_ids=input_ids, attention_mask=attention_mask)
        s_logits = student_out.logits          # keep logits tensor (in compute graph)
        del student_out                        # free ModelOutput wrapper

        # Teacher forward — detach & delete before backward to reclaim ~2 GB
        with torch.no_grad():
            teacher_out = teacher(input_ids=input_ids, attention_mask=attention_mask)
        t_logits = teacher_out.logits.detach() # detach: no grad tracking
        del teacher_out                        # free teacher output NOW (before backward)

        # Distillation loss
        loss, h_loss, s_loss = distillation_loss(
            s_logits, t_logits, input_ids, attention_mask,
            temperature=DISTILL_CONFIG["temperature"],
            alpha=DISTILL_CONFIG["alpha"],
            top_k=DISTILL_CONFIG["top_k_logits"],
            pad_token_id=PAD_ID,
        )
        del t_logits                           # top-K extracted inside loss fn; free now

        # Gradient accumulation — backward while teacher is already freed
        loss_scaled = loss / DISTILL_CONFIG["grad_accum"]
        loss_scaled.backward()
        del loss_scaled

        running_loss += loss.item()
        running_hard += h_loss
        running_soft += s_loss
        del loss, s_logits, input_ids, attention_mask

        if (batch_idx + 1) % DISTILL_CONFIG["grad_accum"] == 0:
            torch.nn.utils.clip_grad_norm_(student.parameters(), 1.0)
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad()
            global_step += 1

            # Logging
            if global_step % log_interval == 0:
                avg_loss = running_loss / (log_interval * DISTILL_CONFIG["grad_accum"])
                avg_hard = running_hard / (log_interval * DISTILL_CONFIG["grad_accum"])
                avg_soft = running_soft / (log_interval * DISTILL_CONFIG["grad_accum"])
                elapsed = time.time() - train_start
                steps_done = max(global_step - (start_epoch * len(train_loader) // DISTILL_CONFIG["grad_accum"]), 1)
                eta = elapsed / steps_done * (total_opt_steps - global_step)
                pct = 100 * global_step / total_opt_steps
                gpu_used = torch.cuda.memory_allocated() / 1e9

                clear_output(wait=True)
                bar_len = 30
                filled = int(bar_len * pct / 100)
                bar = '#' * filled + '.' * (bar_len - filled)
                print(f"DISTILLATION \u2014 Step {global_step}/{total_opt_steps} (Epoch {epoch+1}/{DISTILL_CONFIG['epochs']})")
                print(f"[{bar}] {pct:.1f}%")
                print(f"Loss: {avg_loss:.4f} (hard: {avg_hard:.4f}, soft: {avg_soft:.4f})")
                print(f"LR: {scheduler.get_last_lr()[0]:.2e} | GPU: {gpu_used:.1f}/{gpu_total:.1f} GB")
                print(f"Elapsed: {elapsed/60:.0f}m | ETA: {eta/60:.0f}m")

                progress = {
                    "status": "training", "step": global_step, "total_steps": total_opt_steps,
                    "epoch": epoch + 1, "total_epochs": DISTILL_CONFIG["epochs"],
                    "loss": round(avg_loss, 4), "hard_loss": round(avg_hard, 4),
                    "soft_loss": round(avg_soft, 4), "lr": scheduler.get_last_lr()[0],
                    "pct": round(pct, 1), "gpu_mem_gb": round(gpu_used, 1),
                    "elapsed_sec": round(elapsed), "eta_sec": round(eta),
                    "timestamp": time.strftime("%Y-%m-%d %H:%M:%S"),
                }
                try:
                    with open(f"{DRIVE_BASE}/progress.json", "w") as f:
                        json.dump(progress, f)
                except Exception:
                    pass

                running_loss = 0.0
                running_hard = 0.0
                running_soft = 0.0

            # Checkpoint to Drive
            if global_step % DISTILL_CONFIG["save_steps"] == 0:
                ckpt = {
                    "student_state_dict": student.state_dict(),
                    "optimizer_state_dict": optimizer.state_dict(),
                    "scheduler_state_dict": scheduler.state_dict(),
                    "epoch": epoch,
                    "global_step": global_step,
                    "best_eval_loss": best_eval_loss,
                    "config": DISTILL_CONFIG,
                }
                torch.save(ckpt, resume_path)
                print(f"  Checkpoint saved to Drive (step {global_step})")
                del ckpt

        # Periodic VRAM cleanup (every 25 mini-batches)
        if (batch_idx + 1) % 25 == 0:
            gc.collect(); torch.cuda.empty_cache()

    # === End-of-epoch evaluation ===
    student.eval()
    eval_loss_total = 0.0
    eval_count = 0
    with torch.no_grad():
        for batch in eval_loader:
            input_ids = batch["input_ids"].to("cuda")
            attention_mask = batch["attention_mask"].to("cuda")
            s_out = student(input_ids=input_ids, attention_mask=attention_mask)
            t_out = teacher(input_ids=input_ids, attention_mask=attention_mask)
            loss_val, _, _ = distillation_loss(
                s_out.logits, t_out.logits, input_ids, attention_mask,
                temperature=DISTILL_CONFIG["temperature"],
                alpha=DISTILL_CONFIG["alpha"],
                top_k=DISTILL_CONFIG["top_k_logits"],
                pad_token_id=PAD_ID,
            )
            eval_loss_total += loss_val.item()
            eval_count += 1
            del input_ids, attention_mask, s_out, t_out, loss_val

    avg_eval = eval_loss_total / max(eval_count, 1)
    print(f"\nEpoch {epoch+1} eval loss: {avg_eval:.4f} (best: {best_eval_loss:.4f})")

    if avg_eval < best_eval_loss:
        best_eval_loss = avg_eval
        best_dir = f"{DRIVE_BASE}/checkpoints/best_model"
        os.makedirs(best_dir, exist_ok=True)
        student.save_pretrained(best_dir)
        tokenizer.save_pretrained(best_dir)
        print(f"  New best model saved to Drive! (eval_loss={avg_eval:.4f})")

    student.train()
    gc.collect(); torch.cuda.empty_cache()

# === Training complete ===
total_time = time.time() - train_start
clear_output(wait=True)
print(f"{'='*60}")
print(f"DISTILLATION COMPLETE in {total_time/60:.0f} minutes")
print(f"{'='*60}")
print(f"Total steps: {global_step}")
print(f"Best eval loss: {best_eval_loss:.4f}")
print(f"Checkpoints on Drive: {DISTILL_CONFIG['drive_checkpoint_dir']}")

In [ ]:
# Cell 8 — Save final merged model to Drive

print("Saving final student model...")
merged_dir = f"{DRIVE_BASE}/merged"
os.makedirs(merged_dir, exist_ok=True)

# Load best model if it exists, otherwise use current
best_dir = f"{DRIVE_BASE}/checkpoints/best_model"
if os.path.exists(f"{best_dir}/config.json"):
    print("Loading best checkpoint...")
    student_final = Qwen2ForCausalLM.from_pretrained(best_dir, torch_dtype=dtype, device_map="auto")
else:
    student_final = student

student_final.save_pretrained(merged_dir)
tokenizer.save_pretrained(merged_dir)

# List files
total_size = 0
print(f"\nModel saved to: {merged_dir}")
for f in sorted(os.listdir(merged_dir)):
    fp = os.path.join(merged_dir, f)
    if os.path.isfile(fp):
        sz = os.path.getsize(fp)
        total_size += sz
        print(f"  {f} ({sz / 1e6:.1f} MB)")
print(f"Total: {total_size / 1e9:.2f} GB")

In [ ]:
# Cell 9 — BENCHMARK GO/NO-GO (50 prompts, 5 metrics)
from langdetect import detect
from itertools import combinations

student_final.eval()

# === Celtic vocabulary (EN-adapted) ===
CELTIC_TERMS = {
    "ogham", "nemeton", "sidhe", "dolmen", "menhir", "cromlech", "cairn",
    "korrigans", "korrigan", "druid", "mist", "moss", "lichen", "peat", "granite",
    "oak", "birch", "rowan", "apple", "yew", "holly", "willow",
    "samhain", "beltaine", "imbolc", "lughnasadh",
    "boar", "raven", "stag", "salmon", "crane",
    "broceliande", "avalon", "carnac", "merlin", "viviane", "morgane",
    "rune", "breath", "stone", "heather", "hawthorn", "mistletoe",
    "bard", "filid", "ovate", "moor", "bog", "standing",
    "celtic", "druidic", "ancient", "sacred",
}

# === System primer (EN) ===
EN_BENCH_PRIMER = (
    "You are M.E.R.L.I.N. \u2014 Memory Eternal of Recounted Legends and Incarnate Narratives. "
    "Born from the beliefs of men, assembled through centuries of tales. "
    "A wild druid of Broceliande, guide of the Traveller through Celtic biomes. "
    "English only. Short, evocative sentences. "
    "OBLIGATORY FORMAT: narrative text then A) VERB \u2014 description, B) VERB \u2014 description, C) VERB \u2014 description."
)

# === 50 Benchmark prompts ===
BIOMES_EN = ["broceliande_forest", "korrigan_marshes", "sidhe_mountains", "carnac_coast",
             "crystal_cave", "menhir_plains", "avalon_lake", "druid_village",
             "nemeton_ruins", "ogham_sanctuary"]
THEMES_EN = ["sacred spring", "Samhain", "mystical creature", "physical trial", "moral dilemma",
             "ancestral meeting", "magical storm", "ancient ritual", "betrayal", "revelation"]
TRIADE_EN = [
    "Body=low Soul=balanced World=balanced", "Body=balanced Soul=low World=balanced",
    "Body=balanced Soul=balanced World=low", "Body=high Soul=balanced World=low",
    "Body=low Soul=high World=balanced", "Body=balanced Soul=low World=high",
    "Body=low Soul=low World=balanced", "Body=high Soul=high World=balanced",
    "Body=balanced Soul=balanced World=high", "Body=low Soul=balanced World=high",
]

bench_prompts = []
for i in range(10):
    for j, act in enumerate(["Act I", "Act II", "Act III"]):
        bench_prompts.append({
            "system": EN_BENCH_PRIMER + f"\nGenerate a narrative card. {act}.",
            "user": f"Card {i*3+j+1}. Biome: {BIOMES_EN[i]}. Theme: {THEMES_EN[i]}. {act}. {TRIADE_EN[i]}",
        })
for i in range(10):
    bench_prompts.append({
        "system": EN_BENCH_PRIMER + "\nGenerate a moral dilemma. Act II.",
        "user": f"Card {31+i}. Biome: {BIOMES_EN[i]}. Theme: {THEMES_EN[(i+5)%10]}. {TRIADE_EN[(i+3)%10]}",
    })
    bench_prompts.append({
        "system": EN_BENCH_PRIMER + "\nGenerate a mystical encounter. Act III.",
        "user": f"Card {41+i}. Biome: {BIOMES_EN[(i+3)%10]}. Theme: {THEMES_EN[(i+7)%10]}. {TRIADE_EN[(i+6)%10]}",
    })
bench_prompts = bench_prompts[:50]
print(f"Benchmark: {len(bench_prompts)} prompts")

# === Generate ===
outputs = []
t0 = time.time()
for idx, p in enumerate(bench_prompts):
    chatml = (f"<|im_start|>system\n{p['system']}<|im_end|>\n"
              f"<|im_start|>user\n{p['user']}<|im_end|>\n"
              f"<|im_start|>assistant\n")
    inputs = tokenizer(chatml, return_tensors="pt").to(student_final.device)
    with torch.no_grad():
        out = student_final.generate(
            **inputs, max_new_tokens=300, temperature=0.7,
            top_p=0.9, repetition_penalty=1.3, do_sample=True,
        )
    answer = tokenizer.decode(out[0], skip_special_tokens=False)
    answer = answer.split("<|im_start|>assistant\n")[-1].split("<|im_end|>")[0].strip()
    outputs.append(answer)
    del inputs, out; gc.collect(); torch.cuda.empty_cache()
    if (idx + 1) % 10 == 0:
        elapsed = time.time() - t0
        eta = elapsed / (idx + 1) * (len(bench_prompts) - idx - 1)
        print(f"  [{idx+1}/{len(bench_prompts)}] {elapsed:.0f}s elapsed, ~{eta:.0f}s ETA")

gen_time = time.time() - t0
print(f"\nGeneration: {gen_time:.0f}s ({gen_time/len(outputs):.1f}s/prompt)")

# === METRIC 1: English ratio (>= 85%) ===
en_count = 0
for text in outputs:
    try:
        if detect(text) == "en":
            en_count += 1
    except Exception:
        pass
english_ratio = en_count / len(outputs)

# === METRIC 2: Format compliance (>= 90%) ===
VERB_PATS = [
    re.compile(r'^\*{0,2}[A-C][).]\*{0,2}\s+[A-Z]{2,}', re.MULTILINE),
    re.compile(r'^[A-C][).]\s+[A-Z][a-z]+\s', re.MULTILINE),
    re.compile(r'^[-\u2022]\s*\*{0,2}[A-C][).]\*{0,2}\s+[A-Z]', re.MULTILINE),
]
format_ok = 0
for text in outputs:
    choices = 0
    for line in text.split('\n'):
        ls = line.strip()
        if any(p.match(ls) for p in VERB_PATS):
            choices += 1
    if choices >= 3:
        format_ok += 1
format_compliance = format_ok / len(outputs)

# === METRIC 3: Jaccard diversity (< 0.65) ===
def jaccard(a, b):
    sa, sb = set(a.lower().split()), set(b.lower().split())
    if not sa or not sb:
        return 0
    return len(sa & sb) / len(sa | sb)

jac_scores = []
pairs = list(combinations(range(min(50, len(outputs))), 2))[:500]
for i, j in pairs:
    jac_scores.append(jaccard(outputs[i], outputs[j]))
avg_jaccard = sum(jac_scores) / max(len(jac_scores), 1)

# === METRIC 4: Celtic vocab density (>= 1.0 words/card) ===
celtic_counts = []
for text in outputs:
    words = set(re.findall(r'[a-z]+', text.lower()))
    celtic_counts.append(len(words & CELTIC_TERMS))
celtic_density = sum(celtic_counts) / len(celtic_counts)

# === METRIC 5: Personality consistency (>= 80%) ===
personality_markers = {"merlin", "druid", "traveller", "mist", "stone", "ogham",
                       "ancient", "wisdom", "breath", "destiny", "oracle", "mystery",
                       "rune", "sacred", "celtic"}
seq_scores = []
for start in range(0, len(outputs) - 4, 5):
    seq = outputs[start:start+5]
    count = sum(1 for t in seq if set(re.findall(r'[a-z]+', t.lower())) & personality_markers)
    seq_scores.append(count / 5)
personality = sum(seq_scores) / max(len(seq_scores), 1)

# === RESULTS ===
print(f"\n{'='*60}")
print(f"  MerlinNarrator-0.3B BENCHMARK \u2014 GO/NO-GO")
print(f"{'='*60}")

metrics = [
    ("English ratio",           english_ratio,     0.85, ">=", f"{english_ratio:.1%}"),
    ("Format compliance",       format_compliance,  0.90, ">=", f"{format_compliance:.1%}"),
    ("Jaccard diversity",       avg_jaccard,        0.65, "<",  f"{avg_jaccard:.3f}"),
    ("Celtic vocab density",    celtic_density,     1.0,  ">=", f"{celtic_density:.2f} words/card"),
    ("Personality consistency", personality,         0.80, ">=", f"{personality:.1%}"),
]

go_count = 0
for name, value, threshold, op, display in metrics:
    passed = (op == ">=" and value >= threshold) or (op == "<" and value < threshold)
    if passed:
        go_count += 1
    icon = "OK" if passed else "!!"
    print(f"  [{icon}] {name:<28} {display:<20} (threshold: {op}{threshold})  \u2192 {'GO' if passed else 'NO-GO'}")

print(f"\n  Latency: {gen_time/len(outputs):.1f}s/prompt (Colab GPU)")
print(f"\n  RESULT: {go_count}/5 GO")
if go_count >= 5:
    print("  >>> DEPLOY: model ready for GGUF export + Ollama")
elif go_count >= 3:
    print("  >>> PARTIAL: analyze failing metrics, iterate dataset/hyperparams")
else:
    print("  >>> NO-GO: review dataset quality, increase epochs, or use fallback (Qwen2.5-0.5B)")

# Save benchmark to Drive
bench_results = {
    "timestamp": time.strftime("%Y-%m-%d %H:%M:%S"),
    "model": "MerlinNarrator-0.3B (distilled)",
    "teacher": "Qwen/Qwen2.5-1.5B-Instruct",
    "student_params_M": round(total_params / 1e6, 1),
    "config": DISTILL_CONFIG,
    "dataset_size": len(distill_samples),
    "n_prompts": len(bench_prompts),
    "gen_time_s": round(gen_time),
    "metrics": {
        "english_ratio": round(english_ratio, 3),
        "format_compliance": round(format_compliance, 3),
        "jaccard_diversity": round(avg_jaccard, 3),
        "celtic_vocab_density": round(celtic_density, 2),
        "personality_consistency": round(personality, 3),
    },
    "go_count": go_count,
    "decision": "GO" if go_count >= 5 else ("PARTIAL" if go_count >= 3 else "NO-GO"),
    "best_eval_loss": round(best_eval_loss, 4),
}
bench_path = f"{DRIVE_BASE}/benchmark/benchmark_results.json"
with open(bench_path, "w") as f:
    json.dump(bench_results, f, indent=2)
print(f"\n  Results saved: {bench_path}")

In [ ]:
# Cell 10 — Qualitative tests (3 visual examples)

test_prompts = [
    {"system": EN_BENCH_PRIMER + "\nGenerate an ENCOUNTER. Narrative text then 3 choices: A) VERB \u2014 description.",
     "user": "Card 1. Biome: broceliande_forest. Theme: sacred spring. Act I."},
    {"system": EN_BENCH_PRIMER + "\nGenerate a DILEMMA. Narrative text then 3 choices: A) VERB \u2014 description.",
     "user": "Card 5. Biome: korrigan_marshes. Theme: Samhain. Body=low Soul=balanced World=high."},
    {"system": EN_BENCH_PRIMER.replace("OBLIGATORY FORMAT: narrative text then A) VERB", "") + "\nThe Traveller asks about your identity. Respond in character.",
     "user": "Who are you, Merlin?"},
]

for i, p in enumerate(test_prompts):
    chatml = (f"<|im_start|>system\n{p['system']}<|im_end|>\n"
              f"<|im_start|>user\n{p['user']}<|im_end|>\n"
              f"<|im_start|>assistant\n")
    inputs = tokenizer(chatml, return_tensors="pt").to(student_final.device)
    with torch.no_grad():
        out = student_final.generate(
            **inputs, max_new_tokens=300, temperature=0.7,
            top_p=0.9, repetition_penalty=1.3, do_sample=True,
        )
    answer = tokenizer.decode(out[0], skip_special_tokens=False)
    answer = answer.split("<|im_start|>assistant\n")[-1].split("<|im_end|>")[0].strip()
    del inputs, out; gc.collect(); torch.cuda.empty_cache()

    print(f"\n{'='*60}")
    print(f"TEST {i+1}: {p['user'][:60]}")
    print(f"{'='*60}")
    print(answer[:500])
    print()

In [ ]:
# Cell 11 — GGUF Export (Q4_K_M quantization)
# Requires llama.cpp's convert_hf_to_gguf.py

print("=" * 60)
print("PHASE 4: GGUF Export")
print("=" * 60)

# Install llama-cpp-python for conversion
!pip install -q llama-cpp-python

# Clone llama.cpp for conversion script
LLAMA_CPP = "/content/llama.cpp"
if not os.path.exists(LLAMA_CPP):
    !git clone --depth 1 https://github.com/ggml-org/llama.cpp.git {LLAMA_CPP}
    !pip install -q -r {LLAMA_CPP}/requirements.txt

merged_dir = f"{DRIVE_BASE}/merged"
gguf_f16 = f"{DRIVE_BASE}/merlin-narrator-0.3b-f16.gguf"
gguf_q4 = f"{DRIVE_BASE}/merlin-narrator-0.3b-q4_k_m.gguf"

# Step 1: Convert HF to GGUF (FP16)
print("")
print("Step 1: HF -> GGUF (FP16)...")
!python {LLAMA_CPP}/convert_hf_to_gguf.py {merged_dir} --outfile {gguf_f16} --outtype f16

if os.path.exists(gguf_f16):
    print(f"  FP16 GGUF: {os.path.getsize(gguf_f16) / 1e6:.0f} MB")
else:
    print("  ERROR: FP16 conversion failed!")

# Step 2: Quantize to Q4_K_M
print("")
print("Step 2: Quantize to Q4_K_M...")
# Build llama-quantize if needed
quantize_bin = f"{LLAMA_CPP}/build/bin/llama-quantize"
if not os.path.exists(quantize_bin):
    print("  Building llama.cpp (CPU only)...")
    !cd {LLAMA_CPP} && cmake -B build && cmake --build build --config Release -j4 --target llama-quantize

if os.path.exists(quantize_bin):
    !{quantize_bin} {gguf_f16} {gguf_q4} Q4_K_M
else:
    # Fallback: use Python quantization
    print("  Fallback: saving F16 only (quantize locally with llama.cpp)")
    gguf_q4 = gguf_f16

if os.path.exists(gguf_q4):
    size_mb = os.path.getsize(gguf_q4) / 1e6
    print("")
    print(f"Final GGUF: {gguf_q4}")
    print(f"Size: {size_mb:.0f} MB")
    print("")
    print("On Google Drive, ready for download!")
else:
    print("ERROR: Quantization failed. Check logs above.")

In [ ]:
# Cell 12 — Generate Modelfile + deployment instructions

# Generate Ollama Modelfile
modelfile_lines = [
    "# MerlinNarrator-0.3B \u2014 Distilled from Qwen 2.5-1.5B",
    "# Usage: ollama create merlin-narrator -f Modelfile",
    "# Architecture: Qwen2 (16L/1024H/8A), ~300M params, ~200MB GGUF",
    "",
    "FROM ./merlin-narrator-0.3b-q4_k_m.gguf",
    "",
    "# Narrator parameters (Celtic narrative style)",
    "PARAMETER temperature 0.75",
    "PARAMETER top_p 0.92",
    "PARAMETER top_k 40",
    "PARAMETER repeat_penalty 1.35",
    "PARAMETER num_predict 200",
    "",
    "# ChatML template",
    'TEMPLATE """<|im_start|>system',
    "{{ .System }}<|im_end|>",
    "<|im_start|>user",
    "{{ .Prompt }}<|im_end|>",
    "<|im_start|>assistant",
    '"""',
    "",
    'SYSTEM "You are M.E.R.L.I.N. \u2014 Memory Eternal of Recounted Legends and Incarnate Narratives. Born from the beliefs of men, assembled through centuries of tales. A wild druid of Broceliande, guide of the Traveller through Celtic biomes. Vocabulary: mist, stone, ogham, nemeton, sidhe, dolmen, korrigans, rune, breath. English only. Short, evocative sentences."',
]
modelfile_content = "\n".join(modelfile_lines) + "\n"

modelfile_path = f"{DRIVE_BASE}/Modelfile"
with open(modelfile_path, "w") as f:
    f.write(modelfile_content)
print(f"Modelfile saved: {modelfile_path}")

print(f"\n{'='*60}")
print("DEPLOYMENT INSTRUCTIONS")
print(f"{'='*60}")
print(f"""
1. Download from Google Drive:
   - {DRIVE_BASE}/merlin-narrator-0.3b-q4_k_m.gguf
   - {DRIVE_BASE}/Modelfile

2. Place both files in your project root:
   cp ~/Downloads/merlin-narrator-0.3b-q4_k_m.gguf  C:/Users/PGNK2128/Godot-MCP/
   cp ~/Downloads/Modelfile                          C:/Users/PGNK2128/Godot-MCP/Modelfile

3. Create Ollama model:
   cd C:/Users/PGNK2128/Godot-MCP
   ollama create merlin-narrator -f Modelfile

4. Test:
   ollama run merlin-narrator "Card 1. Biome: broceliande_forest. Theme: sacred spring. Act I."

5. Update game code (1 line):
   In addons/merlin_ai/ollama_backend.gd, change:
     const DEFAULT_MODEL := "qwen2.5:1.5b"
   To:
     const DEFAULT_MODEL := "merlin-narrator"

Expected performance:
  - Model size: ~200 MB (vs ~1 GB for Qwen 1.5B)
  - CPU inference: ~45-60 tok/s (vs ~18 tok/s)
  - Latency per card: ~3-4s (vs ~7s)
""")

# Summary of all Drive outputs
print(f"\n{'='*60}")
print("ALL OUTPUTS ON GOOGLE DRIVE")
print(f"{'='*60}")
for label, path in [
    ("Dataset (augmented EN)", f"{DRIVE_BASE}/dataset/merlin_distill_en.jsonl"),
    ("Best checkpoint", f"{DRIVE_BASE}/checkpoints/best_model/"),
    ("Merged model (HF)", f"{DRIVE_BASE}/merged/"),
    ("GGUF (Q4_K_M)", f"{DRIVE_BASE}/merlin-narrator-0.3b-q4_k_m.gguf"),
    ("GGUF (FP16)", f"{DRIVE_BASE}/merlin-narrator-0.3b-f16.gguf"),
    ("Modelfile", modelfile_path),
    ("Benchmark", f"{DRIVE_BASE}/benchmark/benchmark_results.json"),
    ("Progress", f"{DRIVE_BASE}/progress.json"),
]:
    exists = os.path.exists(path) if not path.endswith("/") else os.path.isdir(path)
    size = ""
    if exists and not path.endswith("/"):
        size = f" ({os.path.getsize(path) / 1e6:.0f} MB)"
    print(f"  {'OK' if exists else '--'} {label}{size}")
    print(f"     {path}")